In [ ]:
!git clone -b dev https://github.com/rafaelTamm/NLE-Project-Spoiler-Detection.git
%cd NLE-Project-Spoiler-Detection

Cloning into 'NLE-Project-Spoiler-Detection'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 74 (delta 30), reused 20 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 31.49 MiB | 18.81 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/NLE-Project-Spoiler-Detection


# Final Results

This notebook combines the exported results of the TF-IDF + Logistic Regression
baseline and ModernBERT. It does not contain manually entered ModernBERT scores.

The main evaluation metric is Macro F1.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results")

modernbert_path = RESULTS_DIR / "modernbert_results.csv"
confidence_path = RESULTS_DIR / "modernbert_confidence.csv"
length_path = RESULTS_DIR / "modernbert_length_analysis.csv"

for path in [modernbert_path, confidence_path, length_path]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run the corresponding export cells in 02_modernbert first."
        )

modernbert = pd.read_csv(modernbert_path)
confidence = pd.read_csv(confidence_path)
length_results = pd.read_csv(length_path)

print("Loaded ModernBERT result files.")

## Baseline results

The notebook looks for an exported TF-IDF result file. This keeps the comparison
notebook independent of manually copied scores.

In [ ]:
baseline_candidates = [
    RESULTS_DIR / "tfidf_results.csv",
    RESULTS_DIR / "baseline_results.csv",
    RESULTS_DIR / "tfidf_baseline_results.csv",
]

baseline_path = next(
    (path for path in baseline_candidates if path.exists()),
    None
)

if baseline_path is None:
    raise FileNotFoundError(
        "No exported TF-IDF results file found. "
        "Export the four baseline experiments from 01_tfidf_baseline_final.ipynb "
        "to results/tfidf_results.csv first."
    )

tfidf = pd.read_csv(baseline_path)

required_columns = {"Experiment", "Macro F1"}

if not required_columns.issubset(tfidf.columns):
    raise ValueError(
        f"{baseline_path} must contain at least: {sorted(required_columns)}"
    )

if not required_columns.issubset(modernbert.columns):
    raise ValueError(
        f"{modernbert_path} must contain at least: {sorted(required_columns)}"
    )

print(f"Loaded baseline results from: {baseline_path}")

## Overall model performance

In [ ]:
experiment_order = [
    "IMDb → IMDb",
    "IMDb → Goodreads",
    "Goodreads → Goodreads",
    "Goodreads → IMDb",
]

tfidf_macro = (
    tfidf[["Experiment", "Macro F1"]]
    .rename(columns={"Macro F1": "TF-IDF"})
)

modernbert_macro = (
    modernbert[["Experiment", "Macro F1"]]
    .rename(columns={"Macro F1": "ModernBERT"})
)

results = tfidf_macro.merge(
    modernbert_macro,
    on="Experiment",
    how="inner"
)

results["Experiment"] = pd.Categorical(
    results["Experiment"],
    categories=experiment_order,
    ordered=True
)

results = results.sort_values("Experiment").reset_index(drop=True)
results["Difference"] = results["ModernBERT"] - results["TF-IDF"]

display(results)

In [ ]:
x = np.arange(len(results))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))

bars_tfidf = ax.bar(
    x - width / 2,
    results["TF-IDF"],
    width,
    label="TF-IDF + Logistic Regression"
)

bars_modernbert = ax.bar(
    x + width / 2,
    results["ModernBERT"],
    width,
    label="ModernBERT"
)

ax.set_ylabel("Macro F1")
ax.set_xlabel("Experiment")
ax.set_title("TF-IDF vs. ModernBERT")
ax.set_xticks(x)
ax.set_xticklabels(results["Experiment"])
ax.set_ylim(0, 1)
ax.legend()

ax.bar_label(bars_tfidf, fmt="%.3f", padding=3)
ax.bar_label(bars_modernbert, fmt="%.3f", padding=3)

plt.tight_layout()
plt.show()

## Cross-domain transfer

For each source domain, the cross-domain Macro F1 is compared with the
corresponding in-domain Macro F1.

In [ ]:
def score_for(model_df, experiment):
    return model_df.loc[
        model_df["Experiment"] == experiment,
        "Macro F1"
    ].iloc[0]

cross_domain_drop = pd.DataFrame([
    {
        "Model": "TF-IDF",
        "Transfer": "IMDb → Goodreads",
        "In-Domain Macro F1": score_for(tfidf, "IMDb → IMDb"),
        "Cross-Domain Macro F1": score_for(tfidf, "IMDb → Goodreads"),
    },
    {
        "Model": "TF-IDF",
        "Transfer": "Goodreads → IMDb",
        "In-Domain Macro F1": score_for(tfidf, "Goodreads → Goodreads"),
        "Cross-Domain Macro F1": score_for(tfidf, "Goodreads → IMDb"),
    },
    {
        "Model": "ModernBERT",
        "Transfer": "IMDb → Goodreads",
        "In-Domain Macro F1": score_for(modernbert, "IMDb → IMDb"),
        "Cross-Domain Macro F1": score_for(modernbert, "IMDb → Goodreads"),
    },
    {
        "Model": "ModernBERT",
        "Transfer": "Goodreads → IMDb",
        "In-Domain Macro F1": score_for(modernbert, "Goodreads → Goodreads"),
        "Cross-Domain Macro F1": score_for(modernbert, "Goodreads → IMDb"),
    },
])

cross_domain_drop["Drop"] = (
    cross_domain_drop["In-Domain Macro F1"]
    - cross_domain_drop["Cross-Domain Macro F1"]
)

display(cross_domain_drop)

## ModernBERT cross-domain confidence

Confidence is the maximum softmax score of the predicted class. It is used here
descriptively and should not be interpreted as calibrated probability.

In [ ]:
display(confidence)

confidence_plot = confidence.copy()
confidence_plot["High Confidence Error Share"] *= 100

fig, ax = plt.subplots(figsize=(7, 4))

bars = ax.bar(
    confidence_plot["Experiment"],
    confidence_plot["High Confidence Error Share"]
)

ax.set_ylabel("Share of errors ≥ 0.90 confidence (%)")
ax.set_title("High-confidence Cross-Domain Errors")
ax.set_ylim(0, 100)
ax.bar_label(bars, fmt="%.1f%%", padding=3)

plt.tight_layout()
plt.show()

## Cross-domain performance by review length

Review length is analyzed descriptively. Because spoiler prevalence changes
substantially with review length, differences between bins must not be interpreted
as a pure causal effect of text length.

In [ ]:
display(length_results)

fig, ax = plt.subplots(figsize=(9, 5))

for experiment, group in length_results.groupby("Experiment", sort=False):
    ax.plot(
        group["Length"],
        group["Macro F1"],
        marker="o",
        label=experiment
    )

ax.set_ylabel("Macro F1")
ax.set_xlabel("Review length in tokens")
ax.set_title("Cross-Domain Macro F1 by Review Length")
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout()
plt.show()

## Automatically derived key results

In [ ]:
mb_imdb_in = score_for(modernbert, "IMDb → IMDb")
mb_imdb_cross = score_for(modernbert, "IMDb → Goodreads")
mb_goodreads_in = score_for(modernbert, "Goodreads → Goodreads")
mb_goodreads_cross = score_for(modernbert, "Goodreads → IMDb")

tfidf_imdb_in = score_for(tfidf, "IMDb → IMDb")
tfidf_imdb_cross = score_for(tfidf, "IMDb → Goodreads")
tfidf_goodreads_in = score_for(tfidf, "Goodreads → Goodreads")
tfidf_goodreads_cross = score_for(tfidf, "Goodreads → IMDb")

print(
    f"ModernBERT IMDb → Goodreads: "
    f"{mb_imdb_in:.4f} → {mb_imdb_cross:.4f} "
    f"(drop: {mb_imdb_in - mb_imdb_cross:.4f})"
)

print(
    f"ModernBERT Goodreads → IMDb: "
    f"{mb_goodreads_in:.4f} → {mb_goodreads_cross:.4f} "
    f"(drop: {mb_goodreads_in - mb_goodreads_cross:.4f})"
)

print(
    f"TF-IDF IMDb → Goodreads: "
    f"{tfidf_imdb_in:.4f} → {tfidf_imdb_cross:.4f} "
    f"(drop: {tfidf_imdb_in - tfidf_imdb_cross:.4f})"
)

print(
    f"TF-IDF Goodreads → IMDb: "
    f"{tfidf_goodreads_in:.4f} → {tfidf_goodreads_cross:.4f} "
    f"(drop: {tfidf_goodreads_in - tfidf_goodreads_cross:.4f})"
)

print()
print("ModernBERT improvement over TF-IDF:")
for _, row in results.iterrows():
    print(
        f"{row['Experiment']}: "
        f"{row['Difference']:+.4f} Macro F1"
    )

## Interpretation

The final interpretation should be based on the values produced above rather than
manually copied scores. The central questions are:

- How much does performance change from in-domain to cross-domain evaluation?
- Is transfer symmetric between IMDb and Goodreads?
- How much does ModernBERT improve over the TF-IDF baseline?
- Are cross-domain errors made with high confidence?
- How is review length associated with cross-domain performance, while accounting
  for the strong relationship between review length and spoiler prevalence?

These observations can be transferred into the Results and Discussion sections of
the report without treating descriptive associations as causal effects.